# Snack Analysis

**Vinith Xavier Madanu**

Compare McDonald's and Starbucks nutrition using PySpark in Google Colab. Run the cells from top to bottom. Results describe the supplied menu records, not current menus or equal-sized portions.


## 1. Environment setup

Install the required packages and start Spark.


In [42]:
# Install PySpark for analysis and openpyxl for reading Excel files.
# The -q option reduces installation messages.

!pip install -q pyspark openpyxl

In [43]:
# Create a Spark session named Snack Analysis.
# local[*] lets Spark use the available CPU cores.
# getOrCreate() reuses a session if one is already running.

# Start Spark
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Snack Analysis")
    .master("local[*]")
    .getOrCreate()
)

print("Spark is ready!")
print("Spark version:", spark.version)

Spark is ready!
Spark version: 4.0.4


## 2. Pulling data from GitHub

Download the repository and check its Excel files.


In [44]:
# Pull the datasets from GitHub into the Colab working folder.
# If the repository already exists, reuse it so rerunning this cell does not fail.

from pathlib import Path
import subprocess

repo_path = Path("/content/snack-analysis")

if not repo_path.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/vinithmadanu/snack-analysis.git", str(repo_path)],
        check=True
    )
else:
    print("Repository already downloaded. Using existing files.")


Repository already downloaded. Using existing files.


In [45]:
# Find the Excel files in the downloaded repository.
# rglob also searches inside subfolders; relative_to displays shorter paths.

from pathlib import Path

repo_path = Path("/content/snack-analysis")

for file in repo_path.rglob("*.xlsx"):
    print(file.relative_to(repo_path))

Mc-Donald-Menu.xlsx
starbucks-menu-nutrition-food.xlsx
starbucks-menu-nutrition-drinks.xlsx
starbucks_drinkMenu_expanded.xlsx


## 3. Loading the datasets

Read Excel with pandas, then create Spark DataFrames for the analysis.


In [46]:
# Import pandas to read the small Excel files before loading them into Spark.

import pandas as pd

In [47]:
# Read all four Excel files from the GitHub repository folder.
# Each pandas DataFrame holds one source dataset.

base_path = "/content/snack-analysis/"

mcd_pd = pd.read_excel(base_path + "Mc-Donald-Menu.xlsx")

sb_expanded_pd = pd.read_excel(base_path + "starbucks_drinkMenu_expanded.xlsx")

sb_drinks_pd = pd.read_excel(base_path + "starbucks-menu-nutrition-drinks.xlsx")

sb_food_pd = pd.read_excel(base_path + "starbucks-menu-nutrition-food.xlsx")

In [48]:
# Name the blank first-column header Item in the two basic Starbucks files.
# Keep all other original column names.

# Give the unnamed product columns a clear name
sb_drinks_pd.rename(columns={sb_drinks_pd.columns[0]: "Item"}, inplace=True)
sb_food_pd.rename(columns={sb_food_pd.columns[0]: "Item"}, inplace=True)

In [49]:
# Define a helper that creates a Spark DataFrame from a pandas DataFrame.
# Load columns as strings first because some Excel columns mix numbers and text.
# Convert missing pandas values to None, which Spark stores as NULL.
# Build an explicit schema so even empty columns have a known type.
# Use the helper to create the four Spark DataFrames.

# Load every column as text first to avoid mixed-data-type errors.
# We will convert nutrition columns to numbers during cleaning.
def to_spark(pdf):
    clean = pdf.astype("string")
    rows = [
        tuple(None if pd.isna(value) else str(value) for value in row)
        for row in clean.itertuples(index=False, name=None)
    ]

    from pyspark.sql.types import StructType, StructField, StringType

    schema = StructType([
        StructField(str(column), StringType(), True)
        for column in clean.columns
    ])

    return spark.createDataFrame(rows, schema)

mcd_df = to_spark(mcd_pd)
sb_expanded_df = to_spark(sb_expanded_pd)
sb_drinks_df = to_spark(sb_drinks_pd)
sb_food_df = to_spark(sb_food_pd)


In [50]:
# Count the rows in each dataset to check that all records loaded.

# Check the number of records
print("McDonald's:", mcd_df.count())
print("Starbucks expanded drinks:", sb_expanded_df.count())
print("Starbucks drinks:", sb_drinks_df.count())
print("Starbucks food:", sb_food_df.count())

McDonald's: 260
Starbucks expanded drinks: 242
Starbucks drinks: 177
Starbucks food: 113


In [51]:
# Preview five records from each food source.
# truncate=False displays the full text of each value.

mcd_df.show(5, truncate=False)
sb_food_df.show(5, truncate=False)

+---------+--------------------------------+--------------+--------+-----------------+---------+-------------------------+-------------+-----------------------------+---------+-----------+---------------------------+------+----------------------+-------------+-----------------------------+-------------+-----------------------------+------+-------+-------------------------+-------------------------+-----------------------+--------------------+
|Category |Item                            |Serving Size  |Calories|Calories from Fat|Total Fat|Total Fat (% Daily Value)|Saturated Fat|Saturated Fat (% Daily Value)|Trans Fat|Cholesterol|Cholesterol (% Daily Value)|Sodium|Sodium (% Daily Value)|Carbohydrates|Carbohydrates (% Daily Value)|Dietary Fiber|Dietary Fiber (% Daily Value)|Sugars|Protein|Vitamin A (% Daily Value)|Vitamin C (% Daily Value)|Calcium (% Daily Value)|Iron (% Daily Value)|
+---------+--------------------------------+--------------+--------+-----------------+---------+----------

## 4. Inspecting the data

Check the original column names and data types.


In [52]:
# Inspect the McDonald's column names and data types before cleaning.

mcd_df.printSchema()

root
 |-- Category: string (nullable = true)
 |-- Item: string (nullable = true)
 |-- Serving Size: string (nullable = true)
 |-- Calories: string (nullable = true)
 |-- Calories from Fat: string (nullable = true)
 |-- Total Fat: string (nullable = true)
 |-- Total Fat (% Daily Value): string (nullable = true)
 |-- Saturated Fat: string (nullable = true)
 |-- Saturated Fat (% Daily Value): string (nullable = true)
 |-- Trans Fat: string (nullable = true)
 |-- Cholesterol: string (nullable = true)
 |-- Cholesterol (% Daily Value): string (nullable = true)
 |-- Sodium: string (nullable = true)
 |-- Sodium (% Daily Value): string (nullable = true)
 |-- Carbohydrates: string (nullable = true)
 |-- Carbohydrates (% Daily Value): string (nullable = true)
 |-- Dietary Fiber: string (nullable = true)
 |-- Dietary Fiber (% Daily Value): string (nullable = true)
 |-- Sugars: string (nullable = true)
 |-- Protein: string (nullable = true)
 |-- Vitamin A (% Daily Value): string (nullable = true)
 

In [53]:
# Inspect the expanded Starbucks drinks schema before cleaning.

sb_expanded_df.printSchema()

root
 |-- Beverage_category: string (nullable = true)
 |-- Beverage: string (nullable = true)
 |-- Beverage_prep: string (nullable = true)
 |-- Calories: string (nullable = true)
 |-- Total Fat (g): string (nullable = true)
 |-- Trans Fat (g): string (nullable = true)
 |-- Saturated Fat (g): string (nullable = true)
 |-- Sodium (mg): string (nullable = true)
 |-- Total Carbohydrates (g): string (nullable = true)
 |-- Cholesterol (mg): string (nullable = true)
 |-- Dietary Fibre (g): string (nullable = true)
 |-- Sugars (g): string (nullable = true)
 |-- Protein (g): string (nullable = true)
 |-- Vitamin A (% DV): string (nullable = true)
 |-- Vitamin C (% DV): string (nullable = true)
 |-- Calcium (% DV): string (nullable = true)
 |-- Iron (% DV): string (nullable = true)
 |-- Caffeine (mg): string (nullable = true)



In [54]:
# Inspect the basic Starbucks drinks schema before cleaning.

sb_drinks_df.printSchema()

root
 |-- Item: string (nullable = true)
 |-- Calories: string (nullable = true)
 |-- Fat (g): string (nullable = true)
 |-- Carb. (g): string (nullable = true)
 |-- Fiber (g): string (nullable = true)
 |-- Protein: string (nullable = true)
 |-- Sodium: string (nullable = true)



In [55]:
# Inspect the Starbucks food schema before cleaning.

sb_food_df.printSchema()

root
 |-- Item: string (nullable = true)
 |-- Calories: string (nullable = true)
 |-- Fat (g): string (nullable = true)
 |-- Carb. (g): string (nullable = true)
 |-- Fiber (g): string (nullable = true)
 |-- Protein (g): string (nullable = true)



## 5. Data cleaning — numeric conversion

Convert nutrition values to numbers while preserving original column names.


In [56]:
# Import expr to write SQL expressions inside PySpark operations.

from pyspark.sql.functions import expr

In [57]:
# Define a reusable function to convert selected columns to double values.
# Double columns can hold decimal numbers for calculations.
# try_cast converts invalid values to NULL instead of raising an error.
# Backticks protect original names containing spaces, dots, or brackets.
# withColumn replaces the column values without renaming the column.

# Convert selected columns to decimal numbers.
# Missing or invalid values become NULL, not zero.
def convert_to_numbers(df, columns):
    for column in columns:
        df = df.withColumn(column, expr(f"try_cast(`{column}` AS DOUBLE)"))
    return df

In [58]:
# Convert McDonald's nutrition columns to numbers.
# Keep Category, Item, and Serving Size as descriptive text.


# McDonald's: keep the three descriptive columns as text
mcd_df = convert_to_numbers(
    mcd_df,
    [c for c in mcd_df.columns
     if c not in ["Category", "Item", "Serving Size"]]
)

In [59]:
# Convert the main Starbucks expanded-drink nutrition columns to numbers.
# Leave caffeine and vitamin percentage fields unchanged; they are not analyzed here.

# Starbucks expanded drinks: convert the main nutrition columns
sb_expanded_df = convert_to_numbers(
    sb_expanded_df,
    [
        "Calories",
        "Total Fat (g)",
        "Trans Fat (g)",
        "Saturated Fat (g)",
        "Sodium (mg)",
        "Total Carbohydrates (g)",
        "Cholesterol (mg)",
        "Dietary Fibre (g)",
        "Sugars (g)",
        "Protein (g)"
    ]
)

In [60]:
# Convert the basic Starbucks drink and food nutrition columns to numbers.
# Keep the Item column as text.

# Starbucks drinks and food: keep Item as text
sb_drinks_df = convert_to_numbers(sb_drinks_df, [c for c in sb_drinks_df.columns if c != "Item"])

sb_food_df = convert_to_numbers(sb_food_df, [c for c in sb_food_df.columns if c != "Item"])

In [61]:
# Confirm that food nutrition columns now have the double data type.
# Preview the converted values to check the result.

sb_food_df.printSchema()
sb_food_df.show(5, truncate=False)

root
 |-- Item: string (nullable = true)
 |-- Calories: double (nullable = true)
 |-- Fat (g): double (nullable = true)
 |-- Carb. (g): double (nullable = true)
 |-- Fiber (g): double (nullable = true)
 |-- Protein (g): double (nullable = true)

+----------------+--------+-------+---------+---------+-----------+
|Item            |Calories|Fat (g)|Carb. (g)|Fiber (g)|Protein (g)|
+----------------+--------+-------+---------+---------+-----------+
|Chonga Bagel    |300.0   |5.0    |50.0     |3.0      |12.0       |
|8-Grain Roll    |380.0   |6.0    |70.0     |7.0      |10.0       |
|Almond Croissant|410.0   |22.0   |45.0     |3.0      |10.0       |
|Apple Fritter   |460.0   |23.0   |56.0     |2.0      |7.0        |
|Banana Nut Bread|420.0   |22.0   |52.0     |2.0      |6.0        |
+----------------+--------+-------+---------+---------+-----------+
only showing top 5 rows


## 6. Data cleaning — missing values

Count missing values, investigate invalid entries, and retain an analysis copy.


In [62]:
# Import col to refer to columns and sum to count missing values.
# Use the name spark_sum to avoid confusing it with Python's sum function.

from pyspark.sql.functions import col, sum as spark_sum

In [63]:
# Define a function that counts NULL values in every column.
# isNull() identifies missing values; casting to int turns True into 1.
# Summing these values gives the number of missing records per column.
# This checks NULL values, not every possible invalid text marker.

def check_missing(df):
    df.select([
        spark_sum(col(f"`{column}`").isNull().cast("int")).alias(column)
        for column in df.columns
    ]).show(truncate=False)

In [64]:
# Display missing-value counts for the McDonald's dataset.

print("McDonald's")
check_missing(mcd_df)

McDonald's
+--------+----+------------+--------+-----------------+---------+-------------------------+-------------+-----------------------------+---------+-----------+---------------------------+------+----------------------+-------------+-----------------------------+-------------+-----------------------------+------+-------+-------------------------+-------------------------+-----------------------+--------------------+
|Category|Item|Serving Size|Calories|Calories from Fat|Total Fat|Total Fat (% Daily Value)|Saturated Fat|Saturated Fat (% Daily Value)|Trans Fat|Cholesterol|Cholesterol (% Daily Value)|Sodium|Sodium (% Daily Value)|Carbohydrates|Carbohydrates (% Daily Value)|Dietary Fiber|Dietary Fiber (% Daily Value)|Sugars|Protein|Vitamin A (% Daily Value)|Vitamin C (% Daily Value)|Calcium (% Daily Value)|Iron (% Daily Value)|
+--------+----+------------+--------+-----------------+---------+-------------------------+-------------+-----------------------------+---------+-----------+

In [65]:
# Display missing-value counts for the expanded Starbucks drinks dataset.

print("Starbucks expanded drinks")
check_missing(sb_expanded_df)

Starbucks expanded drinks
+-----------------+--------+-------------+--------+-------------+-------------+-----------------+-----------+-----------------------+----------------+-----------------+----------+-----------+----------------+----------------+--------------+-----------+-------------+
|Beverage_category|Beverage|Beverage_prep|Calories|Total Fat (g)|Trans Fat (g)|Saturated Fat (g)|Sodium (mg)|Total Carbohydrates (g)|Cholesterol (mg)|Dietary Fibre (g)|Sugars (g)|Protein (g)|Vitamin A (% DV)|Vitamin C (% DV)|Calcium (% DV)|Iron (% DV)|Caffeine (mg)|
+-----------------+--------+-------------+--------+-------------+-------------+-----------------+-----------+-----------------------+----------------+-----------------+----------+-----------+----------------+----------------+--------------+-----------+-------------+
|0                |0       |0            |0       |1            |0            |0                |0          |0                      |0               |0                |0    

In [66]:
# Display missing-value counts for the basic Starbucks drinks dataset.

print("Starbucks drinks")
check_missing(sb_drinks_df)

Starbucks drinks
+----+--------+-------+---------+---------+-------+------+
|Item|Calories|Fat (g)|Carb. (g)|Fiber (g)|Protein|Sodium|
+----+--------+-------+---------+---------+-------+------+
|0   |85      |85     |85       |85       |85     |85    |
+----+--------+-------+---------+---------+-------+------+



In [67]:
# Display missing-value counts for the Starbucks food dataset.

print("Starbucks food")
check_missing(sb_food_df)

Starbucks food
+----+--------+-------+---------+---------+-----------+
|Item|Calories|Fat (g)|Carb. (g)|Fiber (g)|Protein (g)|
+----+--------+-------+---------+---------+-----------+
|0   |0       |0      |0        |0        |0          |
+----+--------+-------+---------+---------+-----------+



In [68]:
# Investigate missing nutrition values before deciding how to handle them.
# First, show the expanded drink whose fat value became NULL.
# Then inspect the original Excel value to see why conversion failed.
# Finally, combine all six NULL checks with AND to identify drinks with no nutrition data.
# Keep the date-formatted fat value as NULL because its original number is unknown.

from pyspark.sql.functions import col

# Find the drink with missing fat
sb_expanded_df.filter(col("`Total Fat (g)`").isNull()).select("Beverage", "Beverage_prep", "Calories", "Total Fat (g)").show(truncate=False)

# Check its original Excel value before the numeric conversion
print("Original non-numeric fat values:")
original_fat = sb_expanded_pd["Total Fat (g)"]

print(sb_expanded_pd.loc[pd.to_numeric(original_fat, errors="coerce").isna(), ["Beverage", "Beverage_prep", "Total Fat (g)"]].to_string(index=False))

# Count drinks where ALL nutrition values are missing
nutrition_columns = ["Calories", "Fat (g)", "Carb. (g)", "Fiber (g)", "Protein", "Sodium"]

all_missing = col(f"`{nutrition_columns[0]}`").isNull()

for column in nutrition_columns[1:]:
    all_missing = all_missing & col(f"`{column}`").isNull()

print("\nDrinks with all nutrition values missing:", sb_drinks_df.filter(all_missing).count())

sb_drinks_df.filter(all_missing).select("Item").show(10, truncate=False)

+--------------------------------------------+-------------+--------+-------------+
|Beverage                                    |Beverage_prep|Calories|Total Fat (g)|
+--------------------------------------------+-------------+--------+-------------+
|Strawberries & Crème (Without Whipped Cream)|Soymilk      |320.0   |NULL         |
+--------------------------------------------+-------------+--------+-------------+

Original non-numeric fat values:
                                    Beverage Beverage_prep       Total Fat (g)
Strawberries & Crème (Without Whipped Cream)       Soymilk 2024-03-02 00:00:00

Drinks with all nutrition values missing: 85
+-----------------------------------------------+
|Item                                           |
+-----------------------------------------------+
|Ombré Pink Drink                               |
|Pink Drink                                     |
|Violet Drink                                   |
|Evolution Fresh™ Cold-Pressed Apple Berry

In [69]:
# Create an analysis copy that excludes drinks missing every nutrition value.
# how="all" removes a row only when all selected columns are NULL.
# Backticks are needed for the dot in Carb. (g).
# Keep the original DataFrame for reference; do not replace missing values with zero.

# Backticks tell Spark to use the exact column names
sb_drinks_analysis_df = sb_drinks_df.dropna(how="all", subset=["`Calories`", "`Fat (g)`", "`Carb. (g)`", "`Fiber (g)`", "`Protein`", "`Sodium`"])

print("Original drinks:", sb_drinks_df.count())
print("Drinks available for analysis:", sb_drinks_analysis_df.count())

Original drinks: 177
Drinks available for analysis: 92


## 7. Exploring categories and separating food and drinks

Inspect the menu categories before making broad comparisons.


In [70]:
# Group McDonald's rows by category and count the records in each group.
# Sort the category names so the output is easy to read.

# McDonald's categories and number of items
mcd_df.groupBy("Category").count().orderBy("Category").show(truncate=False)

+------------------+-----+
|Category          |count|
+------------------+-----+
|Beef & Pork       |15   |
|Beverages         |27   |
|Breakfast         |42   |
|Chicken & Fish    |27   |
|Coffee & Tea      |95   |
|Desserts          |7    |
|Salads            |6    |
|Smoothies & Shakes|28   |
|Snacks & Sides    |13   |
+------------------+-----+



In [71]:
# Group expanded Starbucks drinks by category and count their records.

# Starbucks drink categories and number of items
sb_expanded_df.groupBy("Beverage_category").count().orderBy(
    "Beverage_category"
).show(truncate=False)

+---------------------------------+-----+
|Beverage_category                |count|
+---------------------------------+-----+
|Classic Espresso Drinks          |58   |
|Coffee                           |4    |
|Frappuccino® Blended Coffee      |36   |
|Frappuccino® Blended Crème       |13   |
|Frappuccino® Light Blended Coffee|12   |
|Shaken Iced Beverages            |18   |
|Signature Espresso Drinks        |40   |
|Smoothies                        |9    |
|Tazo® Tea Drinks                 |52   |
+---------------------------------+-----+



In [72]:
# Import col for the food and drink filters below.

from pyspark.sql.functions import col

In [73]:
# Identify the three McDonald's beverage categories.
# isin() keeps rows whose category appears in this list.

# McDonald's drink categories
drink_categories = ["Beverages", "Coffee & Tea", "Smoothies & Shakes"]

# Drinks
mcd_drinks_df = mcd_df.filter(col("Category").isin(drink_categories))

In [74]:
# Use ~ to reverse the category condition and select food rows.
# Check the food and drink counts after splitting McDonald's menu.
# Use expanded Starbucks drinks for the main comparison; do not combine overlapping sources.

# Food: all remaining categories
mcd_food_df = mcd_df.filter(~col("Category").isin(drink_categories))

print("McDonald's food:", mcd_food_df.count())
print("McDonald's drinks:", mcd_drinks_df.count())
print("Starbucks food:", sb_food_df.count())
print("Starbucks expanded drinks:", sb_expanded_df.count())

McDonald's food: 110
McDonald's drinks: 150
Starbucks food: 113
Starbucks expanded drinks: 242


## 8. Analysis — average food nutrition

Compare calories, fat, carbohydrates, fiber, and protein per listed food item.


In [75]:
# Import average and rounding functions for the comparison.
# Use spark_round as an alias to distinguish it from Python's round function.

from pyspark.sql.functions import avg, round as spark_round

In [76]:
# Calculate average nutrition per listed McDonald's food item.
# Round results to two decimal places and label the displayed measures.
# These averages weight each listed row equally, not by sales or serving weight.

print("McDonald's food — average per listed item")

mcd_food_df.select(
    spark_round(avg("Calories"), 2).alias("Average Calories"),
    spark_round(avg("Total Fat"), 2).alias("Average Fat"),
    spark_round(avg("Carbohydrates"), 2).alias("Average Carbs"),
    spark_round(avg("Dietary Fiber"), 2).alias("Average Fiber"),
    spark_round(avg("Protein"), 2).alias("Average Protein")).show()

McDonald's food — average per listed item
+----------------+-----------+-------------+-------------+---------------+
|Average Calories|Average Fat|Average Carbs|Average Fiber|Average Protein|
+----------------+-----------+-------------+-------------+---------------+
|          462.09|      22.94|        43.36|          2.8|          20.78|
+----------------+-----------+-------------+-------------+---------------+



In [77]:
# Calculate the same food measures for Starbucks.
# Escape Carb. (g) with backticks so Spark reads the dot as part of the name.
# Food portions and product types differ between the two menus.

sb_food_df.select(
    spark_round(avg("Calories"), 2).alias("Average Calories"),
    spark_round(avg("Fat (g)"), 2).alias("Average Fat"),
    spark_round(avg("`Carb. (g)`"), 2).alias("Average Carbs"),
    spark_round(avg("Fiber (g)"), 2).alias("Average Fiber"),
    spark_round(avg("Protein (g)"), 2).alias("Average Protein")).show()

+----------------+-----------+-------------+-------------+---------------+
|Average Calories|Average Fat|Average Carbs|Average Fiber|Average Protein|
+----------------+-----------+-------------+-------------+---------------+
|          356.64|      16.35|        41.49|         2.85|          11.47|
+----------------+-----------+-------------+-------------+---------------+



## 9. Analysis — average drink nutrition

Compare the McDonald's drink subset with the expanded Starbucks drinks dataset.


In [78]:
# Calculate average calories, fat, sugar, and protein for McDonald's drinks.
# Each listed drink size or preparation contributes one row.

print("McDonald's drinks — average per listed item")

mcd_drinks_df.select(
    spark_round(avg("Calories"), 2).alias("Average Calories"),
    spark_round(avg("Total Fat"), 2).alias("Average Fat"),
    spark_round(avg("Sugars"), 2).alias("Average Sugar"),
    spark_round(avg("Protein"), 2).alias("Average Protein")).show()

McDonald's drinks — average per listed item
+----------------+-----------+-------------+---------------+
|Average Calories|Average Fat|Average Sugar|Average Protein|
+----------------+-----------+-------------+---------------+
|          299.47|       7.73|        44.64|           7.88|
+----------------+-----------+-------------+---------------+



In [79]:
# Calculate the same drink averages for Starbucks.
# avg() ignores NULL values, so fat uses 241 rows while the other measures use 242.

print("Starbucks drinks — average per listed item")

sb_expanded_df.select(
    spark_round(avg("Calories"), 2).alias("Average Calories"),
    spark_round(avg("Total Fat (g)"), 2).alias("Average Fat"),
    spark_round(avg("Sugars (g)"), 2).alias("Average Sugar"),
    spark_round(avg("Protein (g)"), 2).alias("Average Protein")).show()

Starbucks drinks — average per listed item
+----------------+-----------+-------------+---------------+
|Average Calories|Average Fat|Average Sugar|Average Protein|
+----------------+-----------+-------------+---------------+
|          193.87|        2.9|        32.96|           6.98|
+----------------+-----------+-------------+---------------+



## 10. Analysis — coffee examples

Inspect drink names and compare listed plain-coffee values without assuming equal portions.


In [80]:
# List distinct Starbucks category and beverage-name pairs.
# This helps identify different drink types and misleading category labels.

sb_expanded_df.select(
    "Beverage_category", "Beverage"
).distinct().orderBy(
    "Beverage_category", "Beverage"
).show(100, truncate=False)

+---------------------------------+---------------------------------------------------+
|Beverage_category                |Beverage                                           |
+---------------------------------+---------------------------------------------------+
|Classic Espresso Drinks          |Caffè Americano                                    |
|Classic Espresso Drinks          |Caffè Latte                                        |
|Classic Espresso Drinks          |Caffè Mocha (Without Whipped Cream)                |
|Classic Espresso Drinks          |Cappuccino                                         |
|Classic Espresso Drinks          |Espresso                                           |
|Classic Espresso Drinks          |Skinny Latte (Any Flavour)                         |
|Classic Espresso Drinks          |Vanilla Latte (Or Other Flavoured Latte)           |
|Coffee                           |Brewed Coffee                                      |
|Frappuccino® Blended Coffee    

In [81]:
# Import col for selecting coffee records.

from pyspark.sql.functions import col

In [82]:
# Find McDonald's coffee entries and display their portions and nutrition.
# The name search also returns iced and flavored coffees; it is not a plain-coffee-only filter.

# Find McDonald's plain coffee entries
mcd_df.filter(col("Category") == "Coffee & Tea").filter(col("Item").contains("Coffee")).select( "Item", "Serving Size", "Calories", "Total Fat", "Sugars").show(50, truncate=False)

+---------------------------------------------------------+------------+--------+---------+------+
|Item                                                     |Serving Size|Calories|Total Fat|Sugars|
+---------------------------------------------------------+------------+--------+---------+------+
|Coffee (Small)                                           |12 fl oz cup|0.0     |0.0      |0.0   |
|Coffee (Medium)                                          |16 fl oz cup|0.0     |0.0      |0.0   |
|Coffee (Large)                                           |16 fl oz cup|0.0     |0.0      |0.0   |
|Regular Iced Coffee (Small)                              |16 fl oz cup|140.0   |4.5      |22.0  |
|Regular Iced Coffee (Medium)                             |22 fl oz cup|190.0   |7.0      |30.0  |
|Regular Iced Coffee (Large)                              |32 fl oz cup|270.0   |9.0      |45.0  |
|Caramel Iced Coffee (Small)                              |16 fl oz cup|130.0   |4.5      |21.0  |
|Caramel I

In [83]:
# Select only Starbucks Brewed Coffee and show its preparation labels.
# Size names alone do not establish equal volumes across brands.

# Show Starbucks plain brewed coffee entries
sb_expanded_df.filter(
    col("Beverage") == "Brewed Coffee").select("Beverage", "Beverage_prep","Calories", "Total Fat (g)", "Sugars (g)").show(truncate=False)

+-------------+-------------+--------+-------------+----------+
|Beverage     |Beverage_prep|Calories|Total Fat (g)|Sugars (g)|
+-------------+-------------+--------+-------------+----------+
|Brewed Coffee|Short        |3.0     |0.1          |0.0       |
|Brewed Coffee|Tall         |4.0     |0.1          |0.0       |
|Brewed Coffee|Grande       |5.0     |0.1          |0.0       |
|Brewed Coffee|Venti        |5.0     |0.1          |0.0       |
+-------------+-------------+--------+-------------+----------+



## 11. Data cleaning — duplicate rows

Check exact duplicates and clean the separate basic Starbucks drinks dataset.


In [84]:
# Check each dataset for exact duplicate rows.
# Subtract the distinct row count from the full row count to count extra copies.
# Different sizes or preparations remain separate when any column differs.

# Count rows that repeat every column value
datasets = {
    "McDonald's": mcd_df,
    "Starbucks expanded drinks": sb_expanded_df,
    "Starbucks drinks": sb_drinks_analysis_df,
    "Starbucks food": sb_food_df
}

for name, df in datasets.items():
    duplicates = df.count() - df.dropDuplicates().count()
    print(f"{name}: {duplicates} exact duplicate rows")

McDonald's: 0 exact duplicate rows
Starbucks expanded drinks: 0 exact duplicate rows
Starbucks drinks: 17 exact duplicate rows
Starbucks food: 0 exact duplicate rows


In [85]:
# Remove exact duplicates only from the basic Starbucks drinks analysis copy.
# This reduces its 92 usable rows to 75 unique rows.
# It does not change the expanded-drink dataset or the main comparison averages.

# Remove exact duplicate rows from the analysis copy
sb_drinks_analysis_df = sb_drinks_analysis_df.dropDuplicates()

print("Starbucks drinks after removing duplicates:", sb_drinks_analysis_df.count())

Starbucks drinks after removing duplicates: 75


## 12. Analysis — highest-calorie food items

Inspect large portions that may influence menu averages.


In [86]:
# Sort McDonald's food by calories from highest to lowest.
# Use Item as a second sort key for ties and show five records.
# Large portions help explain the overall average; keep them as valid menu entries.

print("McDonald's — highest-calorie food items")

mcd_food_df.select("Item", "Serving Size", "Calories", "Total Fat", "Protein").orderBy(col("Calories").desc(), col("Item")).show(5, truncate=False)

McDonald's — highest-calorie food items
+------------------------------------------------------------+---------------+--------+---------+-------+
|Item                                                        |Serving Size   |Calories|Total Fat|Protein|
+------------------------------------------------------------+---------------+--------+---------+-------+
|Chicken McNuggets (40 piece)                                |22.8 oz (646 g)|1880.0  |118.0    |87.0   |
|Big Breakfast with Hotcakes (Large Biscuit)                 |15.3 oz (434 g)|1150.0  |60.0     |36.0   |
|Big Breakfast with Hotcakes (Regular Biscuit)               |14.8 oz (420 g)|1090.0  |56.0     |36.0   |
|Big Breakfast with Hotcakes and Egg Whites (Large Biscuit)  |15.4 oz (437 g)|1050.0  |50.0     |35.0   |
|Big Breakfast with Hotcakes and Egg Whites (Regular Biscuit)|14.9 oz (423 g)|990.0   |46.0     |35.0   |
+------------------------------------------------------------+---------------+--------+---------+-------+
only s

In [87]:
# Show the five highest-calorie Starbucks food items for comparison.

print("Starbucks — highest-calorie food items")

sb_food_df.select("Item", "Calories", "Fat (g)", "Protein (g)").orderBy(col("Calories").desc(), col("Item")).show(5, truncate=False)

Starbucks — highest-calorie food items
+------------------------------------------------+--------+-------+-----------+
|Item                                            |Calories|Fat (g)|Protein (g)|
+------------------------------------------------+--------+-------+-----------+
|Lentils & Vegetable Protein Bowl with Brown Rice|650.0   |29.0   |23.0       |
|Green Goddess Avocado Salad                     |620.0   |37.0   |13.0       |
|Roasted Turkey & Dill Havarti Sandwich          |600.0   |32.0   |32.0       |
|Strawberries & Jam Sandwich                     |600.0   |25.0   |19.0       |
|Za'atar Chicken & Lemon Tahini Salad            |570.0   |23.0   |27.0       |
+------------------------------------------------+--------+-------+-----------+
only showing top 5 rows


## 13. Analysis — average and median food calories

Use approximate medians as an additional descriptive measure.


In [88]:
# Import the functions needed to compare averages and approximate medians.

from pyspark.sql.functions import avg, round as spark_round, expr

In [89]:
# Calculate the average and approximate median McDonald's food calories.
# The 0.5 quantile estimates the middle of the ordered calorie values.
# It is less affected by extreme portions, but does not normalize serving sizes.

print("McDonald's food")

mcd_food_df.select(spark_round(avg("Calories"), 2).alias("Average Calories"), expr("percentile_approx(Calories, 0.5)").alias("Median Calories")).show()

McDonald's food
+----------------+---------------+
|Average Calories|Median Calories|
+----------------+---------------+
|          462.09|          440.0|
+----------------+---------------+



In [90]:
# Calculate the same average and approximate median for Starbucks food.

print("Starbucks food")

sb_food_df.select(spark_round(avg("Calories"), 2).alias("Average Calories"), expr("percentile_approx(Calories, 0.5)").alias("Median Calories")).show()

Starbucks food
+----------------+---------------+
|Average Calories|Median Calories|
+----------------+---------------+
|          356.64|          360.0|
+----------------+---------------+



## 14. Findings from the menu averages

Summarize the observed pattern without claiming that every item at one brand is healthier.


Starbucks has lower average calories and fat for the listed food items, and lower average calories, fat, and sugar for drinks. McDonald’s has higher average protein. These results favor Starbucks on the selected measures, but they do not prove that every Starbucks item is healthier.

## 15. Analysis — median drink calories and sugar

Check whether the drink medians support the pattern in the averages.


In [91]:
# Calculate approximate median calories and sugar for McDonald's drinks.
# These results help check whether the pattern also appears beyond the averages.

print("McDonald's drinks")

mcd_drinks_df.select(expr("percentile_approx(`Calories`, 0.5)").alias("Median Calories"), expr("percentile_approx(`Sugars`, 0.5)").alias("Median Sugar (g)")).show()

McDonald's drinks
+---------------+----------------+
|Median Calories|Median Sugar (g)|
+---------------+----------------+
|          270.0|            44.0|
+---------------+----------------+



In [92]:
# Calculate approximate median calories and sugar for Starbucks drinks.
# Compare these with McDonald's while considering different sizes and preparations.

print("Starbucks drinks")

sb_expanded_df.select(expr("percentile_approx(`Calories`, 0.5)").alias("Median Calories"), expr("percentile_approx(`Sugars (g)`, 0.5)").alias("Median Sugar (g)")).show()

Starbucks drinks
+---------------+----------------+
|Median Calories|Median Sugar (g)|
+---------------+----------------+
|          180.0|            32.0|
+---------------+----------------+



## 16. Limitations

- Portions, product types, and drink preparations differ between brands. These are not equal-portion comparisons.
- Every listed record receives equal weight; these are not sales-weighted averages.
- Starbucks food data does not include sugar or sodium, so those food comparisons are unavailable.
- One expanded-drink fat value is stored as a date and remains NULL. The fat average excludes that value.
- McDonald's medium and large plain coffees both list 16 fl oz; this possible source issue has not been corrected without verification.
- The basic Starbucks drinks data is cleaned separately and is not combined with the expanded menu, to avoid overlapping records.
- Approximate medians are estimates from percentile_approx(), not exact interpolated medians.
